# AOD-Net
Li et al. (ICCV 2017). Reformulates the haze model into one learned variable K(x): J = K·I − K + b. Five conv layers, <10 KB of weights, real-time even on CPU. This is the dehazing model used in the team's real-time pipeline.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
print(torch.__version__, '| CUDA:', torch.cuda.is_available())

## Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE     = "/content/drive/MyDrive"
SRC       = f"{DRIVE}/O-HAZE"          # original O-HAZE: hazy/ + GT/
DATA_ROOT = f"{DRIVE}/O-HAZE-1600"     # resized copy, shared by all methods and the evaluation
OUT       = f"{DRIVE}/outputs/aod_net" # dehazed images, same file names as hazy/

MAX_IMAGES = 5             # first N pairs (None = all 45)
RESIZE = 1600              # long side; O-HAZE is ~3000x4000

In [ ]:
#resize hazy + GT once (skipped if DATA_ROOT already exists), same as the other method notebooks
import os, glob
from PIL import Image

def resize_long(img, long_side):
    if long_side <= 0 or max(img.size) <= long_side:
        return img
    s = long_side / max(img.size)
    return img.resize((round(img.size[0] * s), round(img.size[1] * s)), Image.BICUBIC)

if not os.path.isdir(f"{DATA_ROOT}/hazy"):
    os.makedirs(f"{DATA_ROOT}/hazy"); os.makedirs(f"{DATA_ROOT}/GT")
    pairs = list(zip(sorted(glob.glob(f"{SRC}/hazy/*")), sorted(glob.glob(f"{SRC}/GT/*"))))[:MAX_IMAGES]
    for h, g in pairs:
        hazy = resize_long(Image.open(h).convert('RGB'), RESIZE)
        gt   = Image.open(g).convert('RGB').resize(hazy.size, Image.BICUBIC)
        stem = lambda p: os.path.splitext(os.path.basename(p))[0]
        hazy.save(f"{DATA_ROOT}/hazy/{stem(h)}.png"); gt.save(f"{DATA_ROOT}/GT/{stem(g)}.png")

hazy_paths = sorted(glob.glob(f"{DATA_ROOT}/hazy/*"))
os.makedirs(OUT, exist_ok=True)
print(len(hazy_paths), "images:", [os.path.basename(p) for p in hazy_paths])

## Pretrained weights
Two options:
- `public`: original AOD-Net weights from TheFairBear/PyTorch-Image-Dehazing (trained on indoor synthetic NYU haze)
- `team`: Fahim's checkpoint (`aod_net_best.pth` from the team pipeline), once available

In [ ]:
WEIGHTS = "public"         # "public" or "team"
TEAM_CKPT = f"{DRIVE}/checkpoints/aod_net_best.pth"   # path to Fahim's checkpoint (only used if WEIGHTS == "team")

if WEIGHTS == "public":
    !git clone -q --depth 1 https://github.com/TheFairBear/PyTorch-Image-Dehazing.git /content/aod_public
    CKPT = "/content/aod_public/snapshots/dehazer.pth"
else:
    CKPT = TEAM_CKPT
print("weights:", CKPT)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Model
Same architecture and layer names as `models/aod_net.py` in the team pipeline, so Fahim's checkpoint loads directly. The public checkpoint names the layers `e_conv1`…`e_conv5`, so they are renamed to `conv1`…`conv5` when loading.

In [ ]:
class AODNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.relu = nn.ReLU(inplace=True)
        self.conv1 = nn.Conv2d(3, 3, kernel_size=1, padding=0)
        self.conv2 = nn.Conv2d(3, 3, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(6, 3, kernel_size=5, padding=2)
        self.conv4 = nn.Conv2d(6, 3, kernel_size=7, padding=3)
        self.conv5 = nn.Conv2d(12, 3, kernel_size=3, padding=1)
        self.b = 1.0

    def forward(self, x):
        x1 = self.relu(self.conv1(x))
        x2 = self.relu(self.conv2(x1))
        x3 = self.relu(self.conv3(torch.cat([x1, x2], dim=1)))
        x4 = self.relu(self.conv4(torch.cat([x2, x3], dim=1)))
        k = self.relu(self.conv5(torch.cat([x1, x2, x3, x4], dim=1)))   # K(x)
        return torch.clamp(self.relu(k * x - k + self.b), 0, 1)       # J = K*I - K + b


net = AODNet()
state = torch.load(CKPT, map_location='cpu')
state = {k.replace('e_conv', 'conv'): v for k, v in state.items()}   # public checkpoint layer names
net.load_state_dict(state, strict=True)
net = net.to(DEVICE).eval()
print(f"AOD-Net loaded: {sum(p.numel() for p in net.parameters())} params")

## Method

In [ ]:
@torch.no_grad()
def dehaze(hazy):
    """hazy: PIL RGB image -> uint8 RGB array. Input in [0, 1], no normalization (as in training)."""
    x = torch.from_numpy(np.asarray(hazy, dtype=np.float32) / 255).permute(2, 0, 1)[None].to(DEVICE)
    y = net(x)
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    return (y[0].permute(1, 2, 0).cpu().numpy() * 255 + 0.5).astype(np.uint8)

## Run over the dataset

In [ ]:
import time
for p in hazy_paths:
    hazy = Image.open(p).convert('RGB')
    t0 = time.time()
    pred = dehaze(hazy)
    dt = time.time() - t0
    Image.fromarray(pred).save(f"{OUT}/{os.path.basename(p)}")
    print(f"{os.path.basename(p)} | {hazy.size[0]}x{hazy.size[1]} | {dt * 1000:.1f} ms")

## Preview

In [ ]:
for p in hazy_paths[:1]:
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    for a, im, t in zip(ax, [Image.open(p), Image.open(f"{OUT}/{os.path.basename(p)}")], ['hazy', 'AOD-Net']):
        a.imshow(im); a.set_title(t); a.axis('off')
    plt.tight_layout(); plt.show()

## Evaluate
Score the outputs with `dehazing/evaluation` (see `dehaze_eval_colab.ipynb`):
```python
per_image, summary = evaluate(DATA_ROOT, {"aod_net": OUT})
```

To score Fahim's version separately, set `WEIGHTS = "team"` and change `OUT` to `f"{DRIVE}/outputs/aod_net_team"` before running.